In [1]:
# how to create a new database?
import sqlite3
from datetime import datetime, date, timedelta

conn = sqlite3.connect('test_sqlite3/test.db', isolation_level=None)

- connect can't make directory --> you should create it before creating database with connect
- isolation_level should be None --> recommended by python book 2 

In [4]:
# how create tables in database?
conn.execute('create table if not exists test_table (column_1 text, column_2 real, column_3 int)')

- query can be uppercase or lowercase letters
- create table --> makes table
- if not exists --> recommended by python book 2, good practice to avoid error

In [7]:
conn.execute('create table if not exists test_table_2 (column_1 text not null, column_2 int) strict')

- strict --> changes integers that was entered as a string --> '42' -> 42
- column 1 should not be null

In [8]:
# how to list every tables in sqlite3 database?
conn.execute('select name from sqlite_schema where type="table"').fetchall()

[('test_table',), ('test_table_2',)]

In [10]:
conn.execute('select * from sqlite_master where type="table"').fetchall()

[('table',
  'test_table',
  'test_table',
  2,
  'CREATE TABLE test_table (column_1 text, column_2 real, column_3 int)'),
 ('table',
  'test_table_2',
  'test_table_2',
  3,
  'CREATE TABLE test_table_2 (column_1 text not null) strict')]

# what is differenct between sqlite_schema and sqlite_maseter?
- sqlite_master is lagacy one
- sqlite_schema is modern one --> better to be used


In [11]:
# how to get column names of a table?
conn.execute('pragma table_info(test_table)').fetchall()

[(0, 'column_1', 'TEXT', 0, None, 0),
 (1, 'column_2', 'REAL', 0, None, 0),
 (2, 'column_3', 'INT', 0, None, 0)]

In [5]:
conn.execute('pragma table_info(test_table_2)').fetchall()

[(0, 'column_1', 'TEXT', 1, None, 0)]

# meaning of each item in tuple above
- Column position 
> The 0 indicates that the column is second in the table. Column numbers are zero based, like Python list indexes, so the first column is at position 0.
- Name 
> 'column_1' is the name of the column. Remember that SQLite column and table names are case insensitive.
- Data type 
> 'TEXT' is the SQLite data type of the column_1 column. 
- Whether the column is NOT NULL 
> The 0 means False and that the column is not NOT NULL (that is, you can put None values in this column).
- Default value 
> None is the default value inserted if no other value is specified.
- Whether the column is the primary key 
> The 0 means False, meaning this column is not a primary-key column.

In [ ]:
# how to make a column with default value?
# when defining column --> add default value after column name
conn.execute('create table if not exists test_table_2 (column_1 text defualt null)')

In [7]:
conn.execute('pragma table_info(test_table_2)').fetchall()

[(0, 'column_1', 'TEXT', 1, None, 0)]

In [ ]:
# how to set first column to be primary key for table?
# when defining column add primary key after column name
conn.execute('create table if not exists test_table_3 (column_1 text primary key)')

In [10]:
conn.execute('pragma table_info(test_table_3)').fetchall()

[(0, 'column_1', 'TEXT', 0, None, 1)]

In [14]:
# how to add info to only one column?
day = datetime.datetime.today().date()
conn.execute('insert into test_table_3 values (?)', [day])

/tmp/ipykernel_29665/2878336515.py:3: DeprecationWarning: The default date adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  conn.execute('insert into test_table_3 values (?)', [day])


In [15]:
# how to show all data of a table?
conn.execute('select * from test_table_3').fetchall()

[('2026-03-14',)]

In [38]:
# create a table based on my data model
conn.execute('create table if not exists model_2 (date text primary key, study_work_hours real, explanation text)')

In [39]:
conn.execute('pragma table_info(model_2)').fetchall()

[(0, 'date', 'TEXT', 0, None, 1),
 (1, 'study_work_hours', 'REAL', 0, None, 0),
 (2, 'explanation', 'TEXT', 0, None, 0)]

In [ ]:
# add five years from now
# five years is about 2200 days
from datetime import timedelta
first_day = datetime.datetime.today().date()
last_day = first_day + timedelta(days=2200)
days = []
for i in range( (last_day - first_day).days + 1):
    day = first_day + timedelta(days=i)
    day = day.isoformat()
    conn.execute('insert into model_2 values (?, ?, ?)', [day, None, None])


In [42]:
conn.execute('pragma table_info(model_2)').fetchall()

[(0, 'date', 'TEXT', 0, None, 1),
 (1, 'study_work_hours', 'REAL', 0, None, 0),
 (2, 'explanation', 'TEXT', 0, None, 0)]

In [47]:
conn.execute('select * from model_2').fetchone()

('2026-03-14', None, None)

In [21]:
# how add data to a specific row based on primary key
import datetime

hours = 2.5
explanation = 'explain'
date = datetime.datetime.today().date().isoformat()
conn.execute(f'update model_2 set study_work_hours = (?) where date = (?)', [hours, date])
conn.execute(f'update model_2 set explanation = (?) where date = (?)', [explanation, date])
test = conn.execute(f'select * from model_2 where study_work_hours != "None" and explanation != "holiday"').fetchall()

In [22]:
test[0][1]

2.5

In [ ]:
conn.execute(f'select * from model_2').fetchall()

In [2]:
conn.execute('select * from model_2 where explanation != "None"').fetchall()[-1][0]

'2026-03-14'

In [28]:
last_report = datetime.date(2026, 1, 1)
today = datetime.datetime.today().date()

(today - last_report).days

72

In [5]:
conn.execute('select * from sqlite_schema where type="table"').fetchall()

[('table',
  'test_table',
  'test_table',
  2,
  'CREATE TABLE test_table (column_1 text, column_2 real, column_3 int)'),
 ('table',
  'test_table_2',
  'test_table_2',
  3,
  'CREATE TABLE test_table_2 (column_1 text not null) strict'),
 ('table',
  'test_table_3',
  'test_table_3',
  4,
  'CREATE TABLE test_table_3 (column_1 text primary key)'),
 ('table',
  'model',
  'model',
  6,
  'CREATE TABLE model (date text not null primary key, study_work_hours real default null, explanation text not null) strict'),
 ('table',
  'model_2',
  'model_2',
  8,
  'CREATE TABLE model_2 (date text primary key, study_work_hours real, explanation text)')]

In [4]:
conn.execute('create table if not exists model_4 (date text primary key)')
conn.commit()

In [5]:
first_day = datetime.today().date()
conn = sqlite3.connect('test_sqlite3/test.db', isolation_level=None)
for i in range(2200):
    day = first_day + timedelta(days=i)
    day = day.isoformat()
    conn.execute('insert into model_4 values (?)', [day])

In [12]:
conn.commit()

In [21]:
conn.close()

In [7]:
conn.execute('select * from model_4').fetchall()[-1]

('2032-03-22',)

In [8]:
# test for achievement calculation
conn = sqlite3.connect('test_sqlite3/database.db', isolation_level=None)

In [12]:
results = conn.execute('select * from report where explanation != "None"').fetchall()

In [14]:
number_of_days = 0
total_hours = 0
for result in results:
    if result[2] != 'holiday':
        number_of_days += 1
        total_hours  += result[1] 

number_of_days, total_hours

(29, 96.5)